# Construindo e Treinando um LLM a partir do zero

**BERT**: Pre-training of Deep Bidirectional Transformers for Language Understanding
https://arxiv.org/abs/1810.04805


In [ ]:
%pip install torch

In [4]:
# Imports
import re
import math
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from random import *

## Carrega dados

In [ ]:
# Carrega os dados de texto
texto = open('./texto.txt', 'r').read()

In [25]:
print(texto)

'Olá, como vai? Eu sou a Camila.\n'
'Olá, Camila, meu nome é Fernando. Muito prazer.\n'
'Prazer em conhecer você também. Como você está hoje?\n'
'Ótimo. Meu time de futebol venceu a competição.\n'
'Uau, Parabéns Fernando!\n'
'Obrigado Camila.\n'
'Vamos comer uma pizza mais tarde para celebrar?\n'
'Claro. Você recomenda algum restaurante Camila?\n'
'Sim, abriu um restaurante novo e dizem que a pizza de banana é fenomenal.\n'
'Ok. Nos encontramos no restaurante às sete da noite, pode ser?\n'
'Pode sim. Nos vemos mais tarde então.'


## Pré-Processamento dos Dados de Texto e Construção do Vocabulário

In [26]:
# Filtramos caracteres especiais: '.', ',', '?', '!'
sentences = re.sub("[.,!?\\-]", '', texto.lower()).split('\n') 

In [27]:
word_list = list(set(" ".join(sentences).split()))

In [28]:
print(word_list[:10])

["'claro", "então'", 'está', 'no', 'e', 'muito', 'eu', "camila\\n'", 'comer', 'recomenda']


In [29]:
# Dicionário de palavras com os tokens especiais do BERT
word_dict = {'[PAD]': 0, '[CLS]': 1, '[SEP]': 2, '[MASK]': 3}

In [30]:
# Inclui as palavras no dicionário 
for i, w in enumerate(word_list):
    word_dict[w] = i + 4

In [31]:
number_dict = {i: w for i, w in enumerate(word_dict)}

In [32]:
list(number_dict.items())[:10]

[(0, '[PAD]'),
 (1, '[CLS]'),
 (2, '[SEP]'),
 (3, '[MASK]'),
 (4, "'claro"),
 (5, "então'"),
 (6, 'está'),
 (7, 'no'),
 (8, 'e'),
 (9, 'muito')]

In [33]:
# Tamanho do vocabulário
vocab_size = len(word_dict)
print(vocab_size)

70


In [34]:
# Criamos uma lista para os tokens
token_list = list()

# Loop pelas sentenças para criar a lista de tokens
for sentence in sentences:
    arr = [word_dict[s] for s in sentence.split()]
    token_list.append(arr)

In [35]:
token_list

[[67, 55, 53, 10, 25, 47, 11],
 [67, 22, 63, 39, 56, 57, 9, 58],
 [36, 34, 26, 62, 65, 55, 62, 6, 27],
 [35, 63, 20, 69, 30, 54, 47, 16],
 [31, 23, 17],
 [41, 11],
 [24, 12, 21, 45, 50, 52, 28, 68],
 [4, 62, 13, 59, 15, 11],
 [18, 33, 51, 15, 46, 8, 60, 32, 47, 45, 69, 49, 56, 61],
 [43, 14, 19, 7, 15, 66, 64, 37, 38, 42, 44],
 [48, 29, 14, 40, 50, 52, 5]]

## Definição dos Hiperparâmetros

In [36]:
# Hiperparâmetros
batch_size = 6
n_segments = 2
dropout = 0.2

# Comprimento máximo
maxlen = 100 

# Número máximo de tokens que serão previstos
max_pred = 7

# Número de camadas 
n_layers = 6 

# Número de cabeças no multi-head attention
n_heads = 12

# Tamanho da embedding
d_model = 768

# Tamanho da dimensão feedforward: 4 * d_model
d_ff = d_model * 4

# Dimensão de K(=Q)V
d_k = d_v = 64 

# Epochs
NUM_EPOCHS = 50